# Gate 3 — Phoneme Pipeline End-to-End Test

**Purpose:** Verify the complete phoneme conditioning pipeline works on real audio.

**Steps:**
1. Install phonemizer + espeak-ng (system)
2. Install transformers for Wav2Vec2 CTC alignment
3. Download test audio (LibriSpeech native + L2-ARCTIC Indian)
4. Run phoneme pipeline on 5 native + 5 Indian utterances
5. Generate alignment visualizations
6. Validate len(phone_frames) == len(zc1_frames) for each sample

**Stop condition:** If any validation fails → abort.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# 2. Setup paths and environment
import os, sys, json, time, subprocess, types, warnings, shutil
from pathlib import Path

warnings.simplefilter("ignore")

ACCENTEDGE_DIR = "/content/accentedge"
FA_CODEC_DIR   = "/content/FAcodec"
GATE_DIR       = "/content/gate3_artifacts"
DRIVE_BASE     = "/content/drive/MyDrive/accentedge/runs"
SAMPLE_RATE    = 24000
HOP_LENGTH     = 300
FPS            = SAMPLE_RATE // HOP_LENGTH  # 80

def run(cmd, desc="", check=True, timeout=120):
    print(f"\n>>> {desc or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = r.stdout.strip()
    if out:
        print(out[:500])
    if check and r.returncode != 0:
        print(f"FAILED: {r.stderr[:500]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return r

# ── GPU ──
r = run("nvidia-smi --query-gpu=name --format=csv,noheader", "GPU", check=False)
gpu_name = r.stdout.strip()
print(f"GPU: {gpu_name}")

# ── Environment manifest ──
manifest = {"gpu_name": gpu_name, "python_version": sys.version.split()[0],
            "manifest_timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "gate": "gate3_phoneme_pipeline"}

# Get git SHA if available
r = run("cd /content/accentedge && git rev-parse HEAD", "SHA", check=False)
git_sha = r.stdout.strip() or "unknown"
manifest["accentedge_git_sha"] = git_sha
print(f"accentedge SHA: {git_sha}")

os.makedirs(GATE_DIR, exist_ok=True)
with open(f"{GATE_DIR}/environment.json", "w") as f:
    json.dump(manifest, f, indent=2)

In [ ]:
# 3. Clone repos (idempotent)
run("test -d /content/FAcodec || git clone https://github.com/Plachtaa/FAcodec.git /content/FAcodec",
    "clone FAcodec", check=False)
run("test -f /content/FAcodec/modules/__init__.py || touch /content/FAcodec/modules/__init__.py",
    "modules init", check=False)
run("test -d /content/accentedge || git clone --depth 1 https://github.com/yagami009/accentedge.git /content/accentedge",
    "clone accentedge", check=False)

In [ ]:
# 4. Install system dependencies (espeak-ng) + pip deps
!apt-get update -qq
!apt-get install -y -qq espeak-ng > /dev/null 2>&1
!espeak-ng --version 2>&1 | head -1
print("espeak-ng installed")

!pip install -q numpy soundfile librosa scipy jiwer pyyaml einops \
    huggingface-hub phonemizer torchaudio transformers speechbrain \
    faster-whisper pytest pyworld munch plotly datasets
print("Pip dependencies installed")

In [ ]:
# 5. Path setup + mock audiotools
sys.path = [p for p in sys.path if "/content" not in p]
sys.path.insert(0, FA_CODEC_DIR)
sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")
os.environ["PYTHONPATH"] = FA_CODEC_DIR + "/modules:" + os.environ.get("PYTHONPATH", "")
os.chdir(FA_CODEC_DIR)

# Mock audiotools (same pattern as gate1 scripts)
def _make_mock(name):
    m = types.ModuleType(name)
    m.__path__ = []
    m.__package__ = name
    return m

mock_audio = _make_mock("audiotools")
mock_ml = _make_mock("audiotools.ml")
mock_ml.BaseModel = type("BaseModel", (), {"INTERN": [], "EXTERN": []})
mock_audio.ml = mock_ml
mock_audio.AudioSignal = type("AudioSignal", (), {})
mock_audio.STFTParams = type("STFTParams", (), {})
mock_core = _make_mock("audiotools.core")
mock_core.util = _make_mock("audiotools.core.util")
sys.modules["audiotools"] = mock_audio
sys.modules["audiotools.ml"] = mock_ml
sys.modules["audiotools.core"] = mock_core
sys.modules["audiotools.core.util"] = mock_core.util
print("Path setup complete")

In [ ]:
# 6. Download test audio (LibriSpeech native + L2-ARCTIC Indian)
import torchaudio, torch
from datasets import load_dataset

AUDIO_DIR = "/content/test_audio"
NATIVE_DIR = f"{AUDIO_DIR}/native"
INDIAN_DIR = f"{AUDIO_DIR}/indian"
os.makedirs(NATIVE_DIR, exist_ok=True)
os.makedirs(INDIAN_DIR, exist_ok=True)

# ── Native: LibriSpeech test-clean (5 utterances) ──
print("\n=== Downloading LibriSpeech test-clean ===")
libri = torchaudio.datasets.LIBRISPEECH(root=AUDIO_DIR, url="test-clean", download=True)
native_wavs = []
native_texts = {}
for i in range(min(5, len(libri))):
    wav, sr, _, transcript, _, _ = libri[i]
    fname = f"{NATIVE_DIR}/native_{i:03d}.wav"
    torchaudio.save(fname, wav, sr)
    native_wavs.append(fname)
    native_texts[f"native_{i:03d}"] = transcript
    print(f"  {fname}: {transcript[:60]}")

# ── Indian: L2-ARCTIC (5 utterances) ──
print("\n=== Loading L2-ARCTIC Indian speakers ===")
indian_wavs = []
indian_texts = {}
try:
    ds = load_dataset("osCa/L2-ARCTIC", split="train", trust_remote_code=True)
    all_speakers = sorted(set(ds["speaker_id"]))
    import numpy as np
    np.random.seed(42)
    np.random.shuffle(all_speakers)
    indian_speakers = [s for s in all_speakers if s.startswith("HI")][:4]
    if not indian_speakers:
        indian_speakers = all_speakers[:4]
    print(f"  Indian speakers: {indian_speakers}")

    idx = 0
    for spk in indian_speakers:
        rows = [r for r in ds if r["speaker_id"] == spk]
        np.random.shuffle(rows)
        for row in rows[:2]:
            try:
                arr = row["audio"]["array"]
                sr = row["audio"]["sampling_rate"]
                fname = f"{INDIAN_DIR}/indian_{idx:03d}.wav"
                wav_t = torch.from_numpy(arr).float().unsqueeze(0)
                if sr != SAMPLE_RATE:
                    wav_t = torchaudio.functional.resample(wav_t, sr, SAMPLE_RATE)
                torchaudio.save(fname, wav_t, SAMPLE_RATE)
                text = row.get("transcription", row.get("text", ""))
                indian_wavs.append(fname)
                indian_texts[f"indian_{idx:03d}"] = text
                print(f"  {fname}: {text[:60]}")
                idx += 1
                if idx >= 5:
                    break
            except Exception as e:
                print(f"  [WARN] Skip: {e}")
        if idx >= 5:
            break
except Exception as e:
    print(f"  [WARN] L2-ARCTIC download failed: {e}")

# Fallback: if Indian download failed, use synthetic fallback
if len(indian_wavs) < 3:
    print("\n  Using fallback: downloading OpenSLR Indian English samples...")
    fallback_urls = [
        "https://openslr.magicdatatech.com/resources46/SLR46/speaker001_001.wav",
        "https://openslr.magicdatatech.com/resources46/SLR46/speaker002_001.wav",
        "https://openslr.magicdatatech.com/resources46/SLR46/speaker003_001.wav",
    ]
    idx = len(indian_wavs)
    for url in fallback_urls:
        fname = f"{INDIAN_DIR}/indian_{idx:03d}.wav"
        if not os.path.exists(fname):
            r = subprocess.run(["curl", "-L", "-o", fname, url],
                              capture_output=True, text=True, timeout=30)
            if r.returncode == 0 and os.path.getsize(fname) > 1000:
                indian_wavs.append(fname)
                indian_texts[f"indian_{idx:03d}"] = "(Indian English sample)"
                print(f"  Downloaded: {fname}")
                idx += 1

print(f"\nNative: {len(native_wavs)} files, Indian: {len(indian_wavs)} files")

In [ ]:
# 7. Run phoneme pipeline end-to-end + validate
from accentedge.phase1.phoneme_pipeline import PhonemePipeline
from accentedge.codec.facodec import FACodecAdapter
import torch, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("\n>>> Loading FACodec...")
facodec = FACodecAdapter(device="cuda", facodec_ckpt="Plachta/FAcodec")
facodec.freeze()
print("FACodec ready")

print("\n>>> Loading PhonemePipeline...")
phoneme_pipeline = PhonemePipeline(device="cuda", sample_rate=SAMPLE_RATE)
print("PhonemePipeline ready")

# Run pipeline on all samples
all_wavs = native_wavs + indian_wavs
all_texts = {**native_texts, **indian_texts}
corpus_map = {**{w: "native" for w in native_wavs}, **{w: "indian" for w in indian_wavs}}

pipeline_results = []
all_pass = True

for fpath in all_wavs:
    key = Path(fpath).stem
    transcript = all_texts.get(key, "")
    corpus = corpus_map.get(fpath, "unknown")

    # Load audio
    wav, sr = torchaudio.load(fpath)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    wav = wav.mean(dim=0, keepdim=True).float()  # mono

    # ── Phone IDs from pipeline ──
    phone_ids = phoneme_pipeline(transcript, wav)
    phone_frames = phone_ids.shape[-1]

    # ── FACodec encoding ──
    latents = facodec.encode(wav)
    zc1 = latents.content_zc1
    zc1_frames = zc1.shape[-1]

    # ── Validation ──
    frame_match = (phone_frames == zc1_frames)
    if not frame_match:
        all_pass = False

    duration = wav.shape[-1] / SAMPLE_RATE
    pipeline_results.append({
        "key": key, "corpus": corpus, "transcript": transcript[:60],
        "duration_sec": round(duration, 3),
        "phone_frames": int(phone_frames), "zc1_frames": int(zc1_frames),
        "frame_match": bool(frame_match),
        "fps": round(zc1_frames / duration, 1),
    })
    status = "PASS" if frame_match else "FAIL"
    print(f"  [{corpus:6s}] {key}: phones={phone_frames}, zc1={zc1_frames}, fps={pipeline_results[-1]['fps']} → {status}")

print(f"\n{'='*60}")
print(f"All frame-count validations: {'PASS' if all_pass else 'FAIL'}")
print(f"{'='*60}")
if not all_pass:
    raise RuntimeError("Gate 3 FAILED: frame count mismatch detected.")

In [ ]:
# 8. Generate alignment visualizations
import base64, io
import sys
sys.path.insert(0, f"{ACCENTEDGE_DIR}/scripts")
from visualize_alignment import process_utterance, build_html, SR, HOP_LENGTH, FPS

viz_dir = f"{GATE_DIR}/visualizations"
os.makedirs(viz_dir, exist_ok=True)

# Pick 3 representative samples: 1 native, 1 Indian, 1 numbers
viz_samples = []
for r in pipeline_results:
    if r["corpus"] == "native" and not any(v["corpus"] == "native" for v in viz_samples):
        viz_samples.append(r)
    elif r["corpus"] == "indian" and not any(v["corpus"] == "indian" for v in viz_samples):
        viz_samples.append(r)
    if len(viz_samples) >= 3:
        break

# If fewer than 3, fill remaining
if len(viz_samples) < 3:
    for r in pipeline_results:
        if r not in viz_samples:
            viz_samples.append(r)
        if len(viz_samples) >= 3:
            break

panels = []
print(f"\nGenerating {len(viz_samples)} alignment visualizations...")
for vs in viz_samples:
    fpath = None
    for w in all_wavs:
        if Path(w).stem == vs["key"]:
            fpath = w
            break
    transcript = all_texts.get(vs["key"], "")
    panel = process_utterance(fpath, transcript, vs["key"])
    panels.append(panel)

html_path = f"{viz_dir}/alignment_report.html"
build_html(panels, html_path)
print(f"Report: {html_path}")

In [ ]:
# 9. Save results
results = {
    "gate": "gate3_phoneme_pipeline",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "accentedge_git_sha": git_sha,
    "fps": FPS,
    "hop_length": HOP_LENGTH,
    "sample_rate": SAMPLE_RATE,
    "overall_pass": all_pass,
    "thresholds": {"frame_count_match": True},
    "samples": pipeline_results,
}

with open(f"{GATE_DIR}/pipeline_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {GATE_DIR}/pipeline_results.json")

# Save to Drive
drive_out = f"{DRIVE_BASE}/{git_sha}/gate3"
os.makedirs(drive_out, exist_ok=True)
for fname in ["pipeline_results.json", "environment.json"]:
    src = f"{GATE_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_out}/{fname}")
if os.path.exists(f"{viz_dir}/alignment_report.html"):
    shutil.copy2(f"{viz_dir}/alignment_report.html", f"{drive_out}/alignment_report.html")
print(f"Drive: {drive_out}")

if not all_pass:
    raise RuntimeError("GATE 3 FAILED.")

In [ ]:
# 10. Gate 3 complete
print(f"\nGate 3 {'PASSED' if all_pass else 'FAILED'}.")
print(f"Results: {GATE_DIR}/pipeline_results.json")